# BlocksNet Urban Planning ReAct Agent
LangGraph ReAct agent for urban analysis using the blocksnet library.

In [ ]:
# %pip install langgraph langchain-openai langchain-core python-dotenv geopandas -q

In [ ]:
import os, json, pathlib, warnings
warnings.filterwarnings("ignore")
import pandas as pd
import geopandas as gpd
from dotenv import load_dotenv
from langchain_core.messages import BaseMessage, HumanMessage, AIMessage
from langchain_openai import ChatOpenAI
from langchain_core.tools import tool
from langgraph.prebuilt import create_react_agent
from blocksnet.analysis.network import (mean_accessibility, median_accessibility,
    max_accessibility, calculate_connectivity, area_accessibility, land_use_accessibility)
from blocksnet.analysis.provision import (competitive_provision, shared_provision,
    provision_strong_total, provision_weak_total)
from blocksnet.analysis.centrality import services_centrality, population_centrality
from blocksnet.analysis.diversity import shannon_diversity
from blocksnet.analysis.indicators import calculate_density_indicators, calculate_development_indicators
from blocksnet.analysis.services import services_density, services_count, services_collocation
from blocksnet.blocks.aggregation import aggregate_objects
from blocksnet.blocks.assignment import assign_land_use
from blocksnet.relations import generate_adjacency_graph, calculate_distance_matrix
from blocksnet.enums import LandUse
from blocksnet.config import service_types_config

In [ ]:
load_dotenv()
DATA_DIR = pathlib.Path("data")
OUTPUT_DIR = pathlib.Path("outputs")
OUTPUT_DIR.mkdir(exist_ok=True)
DATA_CACHE: dict = {}

In [ ]:
llm = ChatOpenAI(
    base_url=os.environ["FP2MP_CHAT_URL"],
    api_key=os.environ["FP2MP_API_KEY"],
    model="gpt-4o",        # adjust to what endpoint supports
    temperature=0,
    max_tokens=4096,
)

In [ ]:
def _service_columns(blocks: pd.DataFrame) -> list[str]:
    return sorted([c.replace("capacity_", "", 1) for c in blocks.columns if c.startswith("capacity_")])


def _shape(obj) -> str:
    if hasattr(obj, "shape"):
        return str(obj.shape)
    if hasattr(obj, "number_of_nodes") and hasattr(obj, "number_of_edges"):
        return f"nodes={obj.number_of_nodes()}, edges={obj.number_of_edges()}"
    return type(obj).__name__


def _numeric_stats(s: pd.Series) -> str:
    s = pd.to_numeric(s, errors="coerce").dropna()
    if s.empty:
        return "no numeric values"
    return f"min={s.min():.2f}, mean={s.mean():.2f}, median={s.median():.2f}, max={s.max():.2f}"


def _series_from_result(result, name: str, col: str | None = None) -> pd.Series:
    if isinstance(result, pd.Series):
        return result.rename(name)
    if isinstance(result, pd.DataFrame):
        if col is None and name in result.columns:
            col = name
        if col is not None:
            if col not in result.columns:
                raise ValueError(f"column '{col}' not found in result")
            return result[col].rename(name)
        numeric_cols = result.select_dtypes(include="number").columns
        if len(numeric_cols) == 0:
            raise ValueError("result contains no numeric columns")
        return result[numeric_cols[0]].rename(name)
    return pd.Series(result, name=name)


def _save_result(result, path: pathlib.Path) -> None:
    if isinstance(result, (pd.Series, pd.DataFrame)):
        result.to_csv(path)
    else:
        pd.Series(result).to_csv(path)


@tool
def load_blocks() -> str:
    """Load pre-processed blocks with service capacities from data/blocks_with_services.gpkg."""
    try:
        blocks = gpd.read_file(DATA_DIR / "blocks_with_services.gpkg")
        if "land_use" in blocks.columns:
            blocks["land_use"] = blocks["land_use"].apply(
                lambda s: LandUse(s.split(".")[-1].lower()) if isinstance(s, str) else s
            )
        blocks["site_area"] = blocks.geometry.area
        DATA_CACHE["blocks"] = blocks
        land_use_counts = blocks["land_use"].value_counts(dropna=False).to_string() if "land_use" in blocks else "land_use column not found"
        population_stats = _numeric_stats(blocks["population"]) if "population" in blocks else "population column not found"
        services = _service_columns(blocks)
        return (
            f"Loaded blocks: shape={blocks.shape}, CRS={blocks.crs}\n"
            f"Land use counts:\n{land_use_counts}\n"
            f"Population stats: {population_stats}\n"
            f"Service columns ({len(services)}): {services}"
        )
    except Exception as e:
        return f"Error: {e}"


@tool
def load_accessibility_matrix() -> str:
    """Load pre-computed accessibility matrix from data/acc_mx.pickle."""
    try:
        acc_mx = pd.read_pickle(DATA_DIR / "acc_mx.pickle").astype("float32")
        DATA_CACHE["acc_mx"] = acc_mx
        vals = pd.Series(acc_mx.to_numpy().ravel()).replace([float("inf"), -float("inf")], pd.NA).dropna()
        return f"Loaded accessibility matrix: shape={acc_mx.shape}, dtype={acc_mx.dtypes.iloc[0]}, min={vals.min():.2f}, max={vals.max():.2f}, mean={vals.mean():.2f}"
    except Exception as e:
        return f"Error: {e}"


@tool
def list_cached_data() -> str:
    """List currently loaded DATA_CACHE keys and their shapes."""
    try:
        if not DATA_CACHE:
            return "DATA_CACHE is empty."
        return "\n".join([f"{k}: {_shape(v)}" for k, v in DATA_CACHE.items()])
    except Exception as e:
        return f"Error: {e}"


@tool
def list_service_types() -> str:
    """List valid service type names from BlocksNet config and loaded block columns."""
    try:
        names = set()
        if isinstance(service_types_config, dict):
            names.update(map(str, service_types_config.keys()))
        elif hasattr(service_types_config, "service_types"):
            names.update(map(str, service_types_config.service_types.index))
        if "blocks" in DATA_CACHE:
            names.update(_service_columns(DATA_CACHE["blocks"]))
        return "Service types:\n" + ", ".join(sorted(names))
    except Exception as e:
        return f"Error: {e}"


@tool
def get_block_info(block_id: int) -> str:
    """Return all attributes for one block by index/block id."""
    try:
        if "blocks" not in DATA_CACHE:
            return "Error: load_blocks() must be called first."
        blocks = DATA_CACHE["blocks"]
        row = blocks.loc[block_id] if block_id in blocks.index else blocks.iloc[int(block_id)]
        return row.drop(labels=["geometry"], errors="ignore").to_string()
    except Exception as e:
        return f"Error: {e}"

In [ ]:
def _accessibility_summary(result, cache_key: str, path: pathlib.Path, col: str | None = None) -> str:
    selected_col = col or (cache_key if isinstance(result, pd.DataFrame) and cache_key in result.columns else None)
    s = _series_from_result(result, cache_key, col=selected_col)
    DATA_CACHE[cache_key] = result if isinstance(result, pd.DataFrame) else s
    _save_result(DATA_CACHE[cache_key], path)
    top = s.nsmallest(5).to_string()
    bottom = s.nlargest(5).to_string()
    return f"{cache_key}: {_numeric_stats(s)}\nTop-5 most accessible (lowest time):\n{top}\nTop-5 least accessible (highest time):\n{bottom}\nSaved: {path}"


@tool
def compute_mean_accessibility(out: bool = True) -> str:
    """Compute mean accessibility from the pre-loaded accessibility matrix."""
    try:
        if "blocks" not in DATA_CACHE or "acc_mx" not in DATA_CACHE:
            return "Error: load_blocks() and load_accessibility_matrix() must be called first."
        result = mean_accessibility(DATA_CACHE["acc_mx"], out=out)
        return _accessibility_summary(result, "mean_accessibility", OUTPUT_DIR / "mean_accessibility.csv")
    except Exception as e:
        return f"Error: {e}"


@tool
def compute_median_accessibility(out: bool = True) -> str:
    """Compute median accessibility from the pre-loaded accessibility matrix."""
    try:
        if "blocks" not in DATA_CACHE or "acc_mx" not in DATA_CACHE:
            return "Error: load_blocks() and load_accessibility_matrix() must be called first."
        result = median_accessibility(DATA_CACHE["acc_mx"], out=out)
        return _accessibility_summary(result, "median_accessibility", OUTPUT_DIR / "median_accessibility.csv")
    except Exception as e:
        return f"Error: {e}"


@tool
def compute_max_accessibility(out: bool = True) -> str:
    """Compute max accessibility from the pre-loaded accessibility matrix."""
    try:
        if "blocks" not in DATA_CACHE or "acc_mx" not in DATA_CACHE:
            return "Error: load_blocks() and load_accessibility_matrix() must be called first."
        result = max_accessibility(DATA_CACHE["acc_mx"], out=out)
        return _accessibility_summary(result, "max_accessibility", OUTPUT_DIR / "max_accessibility.csv")
    except Exception as e:
        return f"Error: {e}"


@tool
def compute_connectivity(accessibility_key: str = "mean_accessibility") -> str:
    """Compute connectivity from a cached accessibility result."""
    try:
        if accessibility_key not in DATA_CACHE:
            return f"Error: {accessibility_key} not found. Compute accessibility first."
        result = calculate_connectivity(DATA_CACHE[accessibility_key])
        s = _series_from_result(result, "connectivity")
        DATA_CACHE["connectivity"] = s
        path = OUTPUT_DIR / "connectivity.csv"
        _save_result(s, path)
        return f"Connectivity stats: {_numeric_stats(s)}\nTop-5 highest connectivity:\n{s.nlargest(5).to_string()}\nSaved: {path}"
    except Exception as e:
        return f"Error: {e}"


@tool
def compute_land_use_accessibility(land_use: str, out: bool = True) -> str:
    """Compute accessibility to blocks of a selected LandUse enum name."""
    try:
        if "blocks" not in DATA_CACHE or "acc_mx" not in DATA_CACHE:
            return "Error: load_blocks() and load_accessibility_matrix() must be called first."
        lu = LandUse[land_use.upper()]
        result = land_use_accessibility(DATA_CACHE["acc_mx"], DATA_CACHE["blocks"], land_use=lu, out=out)
        key = f"land_use_accessibility_{land_use.lower()}"
        return _accessibility_summary(result, key, OUTPUT_DIR / f"{key}.csv")
    except Exception as e:
        return f"Error: {e}"


@tool
def compute_area_accessibility(out: bool = True) -> str:
    """Compute area-weighted accessibility. Requires site_area from load_blocks()."""
    try:
        if "blocks" not in DATA_CACHE or "acc_mx" not in DATA_CACHE:
            return "Error: load_blocks() and load_accessibility_matrix() must be called first."
        result = area_accessibility(DATA_CACHE["acc_mx"], DATA_CACHE["blocks"], out=out)
        return _accessibility_summary(result, "area_accessibility", OUTPUT_DIR / "area_accessibility.csv")
    except Exception as e:
        return f"Error: {e}"

In [ ]:
def _service_df(service_type: str) -> pd.DataFrame | str:
    if "blocks" not in DATA_CACHE or "acc_mx" not in DATA_CACHE:
        return "Error: load_blocks() and load_accessibility_matrix() must be called first."
    cap_col = f"capacity_{service_type}"
    if cap_col not in DATA_CACHE["blocks"].columns:
        return f"Error: service type '{service_type}' not found"
    service_df = DATA_CACHE["blocks"][["population", cap_col]].copy()
    service_df = service_df.rename(columns={cap_col: "capacity"}).fillna(0)
    service_df["capacity"] = service_df["capacity"].astype(int)
    return service_df


def _service_demand(service_type: str) -> int | None:
    try:
        if isinstance(service_types_config, dict):
            cfg = service_types_config.get(service_type, {})
            return cfg.get("demand") if isinstance(cfg, dict) else None
        return int(service_types_config[service_type]["demand"])
    except Exception:
        return None


def _provision_summary(df: pd.DataFrame, service_type: str, prefix: str) -> str:
    DATA_CACHE[prefix] = df
    provision_path = OUTPUT_DIR / f"{prefix}_{service_type}.csv"
    df.to_csv(provision_path)
    strong = provision_strong_total(df) if {"demand", "demand_within", "demand_without"}.issubset(df.columns) else None
    weak = provision_weak_total(df) if {"demand", "demand_within", "demand_without"}.issubset(df.columns) else None
    provision_col = "provision" if "provision" in df.columns else ("provision_weak" if "provision_weak" in df.columns else df.select_dtypes(include="number").columns[-1])
    s = pd.to_numeric(df[provision_col], errors="coerce").fillna(0)
    fully = int((s >= 1).sum())
    partially = int(((s > 0) & (s < 1)).sum())
    unserved = int((s <= 0).sum())
    top_underserved = s.nsmallest(10).to_string()
    header = (
        f"{prefix} for {service_type}: strong={strong:.2%}, weak={weak:.2%}\n"
        if strong is not None and weak is not None
        else f"{prefix} for {service_type}: provision stats from column '{provision_col}'\n"
    )
    return (
        header
        + f"Blocks: fully={fully}, partially={partially}, unserved={unserved}\n"
        + f"Top underserved blocks:\n{top_underserved}\nSaved: {provision_path}"
    )


@tool
def compute_service_provision(service_type: str, accessibility_minutes: int = 15, max_depth: int = 1) -> str:
    """Compute competitive provision for one service type and save provision results."""
    try:
        service_df = _service_df(service_type)
        if isinstance(service_df, str):
            return service_df
        demand = _service_demand(service_type)
        result = competitive_provision(service_df, DATA_CACHE["acc_mx"], accessibility_minutes, demand=demand, max_depth=max_depth)
        if isinstance(result, tuple):
            provision_df = result[0]
            for i, item in enumerate(result[1:], start=1):
                if isinstance(item, (pd.Series, pd.DataFrame)):
                    item.to_csv(OUTPUT_DIR / f"competitive_provision_{service_type}_links_{i}.csv")
        else:
            provision_df = result
        return _provision_summary(provision_df, service_type, "competitive_provision")
    except Exception as e:
        return f"Error: {e}"


@tool
def compute_shared_provision(service_type: str, accessibility_minutes: int = 15) -> str:
    """Compute shared provision for one service type and save provision results."""
    try:
        service_df = _service_df(service_type)
        if isinstance(service_df, str):
            return service_df
        result = shared_provision(service_df, DATA_CACHE["acc_mx"], accessibility_minutes)
        provision_df = result[0] if isinstance(result, tuple) else result
        return _provision_summary(provision_df, service_type, "shared_provision")
    except Exception as e:
        return f"Error: {e}"

In [ ]:
def _cache_save_summarize(result, key: str, filename: str, top_label: str = "Top values", col: str | None = None) -> str:
    DATA_CACHE[key] = result
    path = OUTPUT_DIR / filename
    _save_result(result, path)
    if isinstance(result, pd.DataFrame):
        if col is None and key in result.columns:
            col = key
        if col is not None:
            if col not in result.columns:
                raise ValueError(f"column '{col}' not found in result")
            s = result[col]
            return f"{key}: {_numeric_stats(s)}\n{top_label}:\n{s.nlargest(10).to_string()}\nSaved: {path}"
        numeric_cols = result.select_dtypes(include="number").columns
        if len(numeric_cols) == 0:
            return f"{key}: shape={result.shape}. Saved: {path}"
        s = result[numeric_cols[-1]]
    else:
        s = _series_from_result(result, key)
    return f"{key}: {_numeric_stats(s)}\n{top_label}:\n{s.nlargest(10).to_string()}\nSaved: {path}"


@tool
def compute_services_density() -> str:
    """Compute service density for loaded blocks."""
    try:
        if "blocks" not in DATA_CACHE:
            return "Error: load_blocks() must be called first."
        result = services_density(DATA_CACHE["blocks"])
        return _cache_save_summarize(result, "services_density", "services_density.csv")
    except Exception as e:
        return f"Error: {e}"


@tool
def compute_services_count() -> str:
    """Compute service count for loaded blocks."""
    try:
        if "blocks" not in DATA_CACHE:
            return "Error: load_blocks() must be called first."
        result = services_count(DATA_CACHE["blocks"])
        return _cache_save_summarize(result, "services_count", "services_count.csv")
    except Exception as e:
        return f"Error: {e}"


@tool
def compute_services_collocation() -> str:
    """Compute service collocation for loaded blocks."""
    try:
        if "blocks" not in DATA_CACHE:
            return "Error: load_blocks() must be called first."
        result = services_collocation(DATA_CACHE["blocks"])
        return _cache_save_summarize(result, "services_collocation", "services_collocation.csv")
    except Exception as e:
        return f"Error: {e}"


@tool
def compute_shannon_diversity() -> str:
    """Compute Shannon diversity index for loaded blocks."""
    try:
        if "blocks" not in DATA_CACHE:
            return "Error: load_blocks() must be called first."
        result = shannon_diversity(DATA_CACHE["blocks"])
        s = _series_from_result(result, "shannon_diversity", col="shannon_diversity")
        DATA_CACHE["shannon_diversity"] = s
        path = OUTPUT_DIR / "shannon_diversity.csv"
        s.to_csv(path)
        return f"Shannon diversity: {_numeric_stats(s)}\nTop-5 diverse blocks:\n{s.nlargest(5).to_string()}\nBottom-5 diverse blocks:\n{s.nsmallest(5).to_string()}\nSaved: {path}"
    except Exception as e:
        return f"Error: {e}"


@tool
def compute_services_centrality() -> str:
    """Compute service centrality. Requires blocks and accessibility matrix."""
    try:
        if "blocks" not in DATA_CACHE or "acc_mx" not in DATA_CACHE:
            return "Error: load_blocks() and load_accessibility_matrix() must be called first."
        result = services_centrality(DATA_CACHE["acc_mx"], DATA_CACHE["blocks"])
        return _cache_save_summarize(result, "services_centrality", "services_centrality.csv", "Top-10 most central blocks")
    except Exception as e:
        return f"Error: {e}"


@tool
def compute_population_centrality() -> str:
    """Compute population centrality. Requires blocks and a cached adjacency graph."""
    try:
        if "blocks" not in DATA_CACHE or "adjacency_graph" not in DATA_CACHE:
            return "Error: load_blocks() and build_adjacency_graph() must be called first."
        result = population_centrality(DATA_CACHE["blocks"], DATA_CACHE["adjacency_graph"])
        return _cache_save_summarize(result, "population_centrality", "population_centrality.csv", "Top-10 most central blocks", col="population_centrality")
    except Exception as e:
        return f"Error: {e}"

In [ ]:
@tool
def compute_density_indicators() -> str:
    """Compute density indicators and summarize mean FSI/GSI/L by land use."""
    try:
        if "blocks" not in DATA_CACHE:
            return "Error: load_blocks() must be called first."
        result = calculate_density_indicators(DATA_CACHE["blocks"])
        if "land_use" not in result.columns and "land_use" in DATA_CACHE["blocks"].columns:
            result = result.join(DATA_CACHE["blocks"][["land_use"]])
        if "land_use" in result.columns:
            result["land_use"] = result["land_use"].astype(str)
        DATA_CACHE["density_indicators"] = result
        path = OUTPUT_DIR / "density_indicators.csv"
        result.to_csv(path)
        cols = [c for c in ["land_use", "fsi", "gsi", "l", "FSI", "GSI", "L"] if c in result.columns]
        table = result[cols].groupby("land_use").mean(numeric_only=True).to_string() if "land_use" in cols else result.describe().to_string()
        return f"Density indicators saved: {path}\nPer-land-use means / summary:\n{table}"
    except Exception as e:
        return f"Error: {e}"


@tool
def compute_development_indicators() -> str:
    """Compute development indicators for loaded blocks."""
    try:
        if "blocks" not in DATA_CACHE:
            return "Error: load_blocks() must be called first."
        base = DATA_CACHE.get("density_indicators")
        if base is None:
            base = calculate_density_indicators(DATA_CACHE["blocks"])
            DATA_CACHE["density_indicators"] = base
        result = calculate_development_indicators(base)
        DATA_CACHE["development_indicators"] = result
        path = OUTPUT_DIR / "development_indicators.csv"
        result.to_csv(path)
        return f"Development indicators: shape={result.shape}. Saved: {path}\n{result.describe().to_string()}"
    except Exception as e:
        return f"Error: {e}"


@tool
def build_adjacency_graph(buffer_size: int = 0) -> str:
    """Build an adjacency graph for blocks. Takes 20-60 seconds for 3113 blocks."""
    try:
        if "blocks" not in DATA_CACHE:
            return "Error: load_blocks() must be called first."
        graph = generate_adjacency_graph(DATA_CACHE["blocks"], buffer_size=buffer_size)
        DATA_CACHE["adjacency_graph"] = graph
        degrees = dict(graph.degree())
        avg_degree = sum(degrees.values()) / len(degrees) if degrees else 0
        return f"Adjacency graph: nodes={graph.number_of_nodes()}, edges={graph.number_of_edges()}, avg_degree={avg_degree:.2f}"
    except Exception as e:
        return f"Error: {e}"


@tool
def compute_distance_matrix() -> str:
    """Compute Euclidean distance matrix for loaded blocks."""
    try:
        if "blocks" not in DATA_CACHE:
            return "Error: load_blocks() must be called first."
        mx = calculate_distance_matrix(DATA_CACHE["blocks"])
        DATA_CACHE["distance_matrix"] = mx
        path = OUTPUT_DIR / "distance_matrix.csv"
        mx.to_csv(path)
        vals = pd.Series(mx.to_numpy().ravel()).dropna()
        return f"Distance matrix: shape={mx.shape}, mean={vals.mean():.2f}, max={vals.max():.2f}. Saved: {path}"
    except Exception as e:
        return f"Error: {e}"

In [ ]:
SYSTEM_PROMPT = """You are an urban planning analyst with the BlocksNet library.
Rules:
1. Call load_blocks() and load_accessibility_matrix() before analysis tools.
2. Use list_cached_data() to check what's loaded.
3. Use list_service_types() to find valid service names.
4. Return quantitative results with interpretation.
5. Catch and report errors gracefully."""

all_tools = [load_blocks, load_accessibility_matrix, list_cached_data,
             list_service_types, get_block_info,
             compute_mean_accessibility, compute_median_accessibility,
             compute_max_accessibility, compute_connectivity,
             compute_land_use_accessibility, compute_area_accessibility,
             compute_service_provision, compute_shared_provision,
             compute_services_density, compute_services_count,
             compute_services_collocation, compute_shannon_diversity,
             compute_services_centrality, compute_population_centrality,
             compute_density_indicators, compute_development_indicators,
             build_adjacency_graph, compute_distance_matrix]

agent = create_react_agent(model=llm, tools=all_tools, prompt=SYSTEM_PROMPT)

In [ ]:
def run_agent(task: str) -> dict:
    try:
        result = agent.invoke({"messages": [HumanMessage(content=task)]})
        messages: list[BaseMessage] = result["messages"]
        log = [m for m in messages if isinstance(m, (HumanMessage, AIMessage))]
        final_output = next(
            (m.content for m in reversed(messages) if isinstance(m, AIMessage) and m.content),
            "No output generated."
        )
    except Exception as e:
        final_output = f"Error while running agent: {e}"
        log = [HumanMessage(content=task), AIMessage(content=final_output)]
    return {"input": task, "output": final_output, "log": log}

## Task 1: Accessibility Overview

In [ ]:
result1 = run_agent(
    "Проанализируй доступность городских кварталов. "
    "Какие кварталы наиболее и наименее доступны?"
)

print("INPUT:", result1["input"])
print("\nOUTPUT:", result1["output"])
print(f"\nLOG ({len(result1['log'])} messages):")
for msg in result1["log"]:
    role = "USER" if isinstance(msg, HumanMessage) else "AGENT"
    print(f"  [{role}]: {str(msg.content)[:300]}")

## Task 2: School Provision

In [ ]:
result2 = run_agent(
    "Оцени обеспеченность школами. Какие районы испытывают дефицит? "
    "Используй порог доступности 15 минут."
)

print("INPUT:", result2["input"])
print("\nOUTPUT:", result2["output"])
print(f"\nLOG ({len(result2['log'])} messages):")
for msg in result2["log"]:
    role = "USER" if isinstance(msg, HumanMessage) else "AGENT"
    print(f"  [{role}]: {str(msg.content)[:300]}")

## Task 3: Service Diversity Analysis

In [ ]:
result3 = run_agent(
    "Вычисли индекс разнообразия Шеннона для кварталов. "
    "Какие кварталы имеют наиболее и наименее разнообразный набор сервисов?"
)

print("INPUT:", result3["input"])
print("\nOUTPUT:", result3["output"])
print(f"\nLOG ({len(result3['log'])} messages):")
for msg in result3["log"]:
    role = "USER" if isinstance(msg, HumanMessage) else "AGENT"
    print(f"  [{role}]: {str(msg.content)[:300]}")

In [ ]:
# Show saved CSV files
for f in sorted(OUTPUT_DIR.glob("*.csv")):
    df = pd.read_csv(f)
    print(f"\n=== {f.name} ({df.shape}) ===")
    display(df.head(3))